# 07 — Final Model Explainability

This notebook continues directly from:

- **Notebook 03**, which created the identity-aware train–test split;
- **Notebook 04**, which optimized the strongest model families and saved the
  final tuned pipeline;
- **Notebook 05**, which measured univariate associations;
- **Notebook 06**, which estimated adjusted linear associations.

## Explainability question

> **Which facial and image characteristics does the final optimized nonlinear
> model rely on when predicting CR-FIQA scores?**

The notebook explains the exact model selected in Notebook 04 on the unchanged
test set. It includes:

- reproduction of the final test metrics;
- repeated permutation importance;
- global SHAP importance;
- SHAP beeswarm and dependence plots;
- local explanations for a well-predicted and a poorly predicted image;
- aggregation of transformed features back to original project features;
- comparison of RQ2, RQ3, permutation importance, and SHAP.

The notebook intentionally does not perform demographic consistency or fairness
testing. Those analyses belong in a separate notebook.

## Main outputs

```text
results/07_model_explainability/
├── figures/
└── tables/
```

## 1. Shared project setup

In [ ]:
from pathlib import Path

setup_candidates = [
    Path.cwd() / "00_colab_setup.ipynb",
    Path.cwd() / "notebooks" / "00_colab_setup.ipynb",
    Path.cwd().parent / "notebooks" / "00_colab_setup.ipynb",
    Path("/content/drive/MyDrive/FIQA_Project/notebooks/00_colab_setup.ipynb"),
    Path("/content/drive/MyDrive/FIQA_Project/00_colab_setup.ipynb"),
]

SETUP_NOTEBOOK = next(
    (path for path in setup_candidates if path.exists()),
    None,
)

if SETUP_NOTEBOOK is None:
    checked_paths = "\n".join(f"- {path}" for path in setup_candidates)
    raise FileNotFoundError(
        "00_colab_setup.ipynb could not be found.\n"
        "Keep the notebooks in the same notebooks/ directory or update "
        "setup_candidates.\n\n"
        f"Checked:\n{checked_paths}"
    )

print(f"Running setup notebook: {SETUP_NOTEBOOK}")
get_ipython().run_line_magic("run", f'"{SETUP_NOTEBOOK}"')

## 2. Install and import required packages

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("shap") is None:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "shap",
        ]
    )
else:
    print("SHAP is already installed.")

In [ ]:
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
PERMUTATION_REPEATS = 20
MAX_SHAP_ROWS = 1500
TOP_N_FEATURES = 12
TOP_N_DEPENDENCE_PLOTS = 4

TARGET = "cr_fiqa_score"
IMAGE_COLUMN = "index"
IDENTITY_COLUMN = "cls"
GROUP_COLUMN = "group"

## 3. Input and output paths

In [ ]:
DATA_FILE = PROJECT_PATH / "diveface_fiqa_merged.csv"

NOTEBOOK_03_RESULTS = PROJECT_PATH / "results" / "03_ml_models"
NOTEBOOK_03_TABLES = NOTEBOOK_03_RESULTS / "tables"
SPLIT_FILE = NOTEBOOK_03_TABLES / "identity_aware_split.csv"
MODEL_METADATA_FILE = NOTEBOOK_03_RESULTS / "run_metadata.json"

NOTEBOOK_04_RESULTS = (
    PROJECT_PATH
    / "results"
    / "04_model_optimization"
)
NOTEBOOK_04_TABLES = NOTEBOOK_04_RESULTS / "tables"
MODEL_FILE = (
    PROJECT_PATH
    / "models"
    / "04_model_optimization"
    / "best_tuned_model.joblib"
)
OPTIMIZATION_SUMMARY_FILE = (
    NOTEBOOK_04_RESULTS
    / "model_optimization_summary.json"
)

RQ2_FILE = (
    PROJECT_PATH
    / "results"
    / "05_rq2_correlation_analysis"
    / "tables"
    / "pearson_spearman_comparison.csv"
)

RQ3_FILE = (
    PROJECT_PATH
    / "results"
    / "06_rq3_regression_analysis"
    / "tables"
    / "hc3_robust_coefficients.csv"
)

RESULTS_PATH = (
    PROJECT_PATH
    / "results"
    / "07_model_explainability"
)
FIGURES_PATH = RESULTS_PATH / "figures"
TABLES_PATH = RESULTS_PATH / "tables"

for path in [RESULTS_PATH, FIGURES_PATH, TABLES_PATH]:
    path.mkdir(parents=True, exist_ok=True)

required_files = [
    DATA_FILE,
    SPLIT_FILE,
    MODEL_METADATA_FILE,
    MODEL_FILE,
    OPTIMIZATION_SUMMARY_FILE,
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    missing_text = "\n".join(f"- {path}" for path in missing_files)
    raise FileNotFoundError(
        "Required inputs are missing. Run Notebooks 03 and 04 first.\n"
        f"{missing_text}"
    )

print(f"Dataset:      {DATA_FILE}")
print(f"Final model:  {MODEL_FILE}")
print(f"Results:      {RESULTS_PATH}")

## 4. Load the final tuned model and reproduce the exact test split

The test set is not recreated. The row-level split saved by Notebook 03 is
joined back to the same merged dataset. This ensures that Notebook 07 explains
the same observations used for the final evaluation in Notebook 04.

In [ ]:
with MODEL_METADATA_FILE.open("r", encoding="utf-8") as file:
    model_metadata = json.load(file)

with OPTIMIZATION_SUMMARY_FILE.open("r", encoding="utf-8") as file:
    optimization_summary = json.load(file)

feature_columns = model_metadata["feature_columns"]

df = pd.read_csv(DATA_FILE)
split_information = pd.read_csv(SPLIT_FILE)
final_model = joblib.load(MODEL_FILE)

required_dataset_columns = set(
    feature_columns
    + [
        TARGET,
        IMAGE_COLUMN,
        IDENTITY_COLUMN,
        GROUP_COLUMN,
    ]
)

missing_dataset_columns = sorted(
    required_dataset_columns.difference(df.columns)
)

if missing_dataset_columns:
    raise KeyError(
        "The merged dataset is missing required columns: "
        f"{missing_dataset_columns}"
    )

model_df = df[
    [
        IMAGE_COLUMN,
        IDENTITY_COLUMN,
        GROUP_COLUMN,
        TARGET,
    ]
    + feature_columns
].copy()

numeric_features = [
    feature
    for feature in feature_columns
    if feature != GROUP_COLUMN
]

for column in numeric_features + [TARGET]:
    model_df[column] = pd.to_numeric(
        model_df[column],
        errors="coerce",
    )

model_df = model_df.dropna(
    subset=[IDENTITY_COLUMN, TARGET] + feature_columns
).copy()

model_df["source_dataframe_index"] = model_df.index

data_with_split = model_df.merge(
    split_information[
        ["source_dataframe_index", "split"]
    ],
    on="source_dataframe_index",
    how="inner",
    validate="one_to_one",
)

if len(data_with_split) != len(split_information):
    raise RuntimeError(
        "The saved split could not be matched exactly to the current dataset."
    )

test_df = (
    data_with_split.loc[
        data_with_split["split"] == "test"
    ]
    .copy()
    .reset_index(drop=True)
)

X_test = test_df[feature_columns].copy()
y_test = test_df[TARGET].astype(float).copy()

test_metadata = test_df[
    [
        IMAGE_COLUMN,
        IDENTITY_COLUMN,
        GROUP_COLUMN,
        "source_dataframe_index",
    ]
].copy()

expected_test_rows = optimization_summary.get("test_rows")

if expected_test_rows is not None and len(X_test) != expected_test_rows:
    raise RuntimeError(
        "The reconstructed test set differs from Notebook 04: "
        f"{len(X_test)} != {expected_test_rows}"
    )

print(f"Final model type: {type(final_model)}")
print(f"Final model name: {optimization_summary.get('final_model')}")
print(f"Test rows:        {len(X_test):,}")
print(f"Input features:   {len(feature_columns)}")

## 5. Reproduce final-model performance

In [ ]:
test_predictions = final_model.predict(X_test)

test_rmse = mean_squared_error(
    y_test,
    test_predictions,
) ** 0.5

test_mae = mean_absolute_error(
    y_test,
    test_predictions,
)

test_r2 = r2_score(
    y_test,
    test_predictions,
)

performance_summary = pd.DataFrame(
    {
        "Metric": [
            "RMSE",
            "MAE",
            "R_squared",
            "N_test",
        ],
        "Value": [
            test_rmse,
            test_mae,
            test_r2,
            len(X_test),
        ],
    }
)

display(performance_summary)

In [ ]:
reported_metrics = optimization_summary.get(
    "final_test_metrics",
    {},
)

metric_reproduction = pd.DataFrame(
    {
        "Metric": ["RMSE", "MAE", "R_squared"],
        "Notebook_04": [
            reported_metrics.get("rmse"),
            reported_metrics.get("mae"),
            reported_metrics.get("r2"),
        ],
        "Notebook_07": [
            test_rmse,
            test_mae,
            test_r2,
        ],
    }
)

metric_reproduction["Absolute_Difference"] = (
    metric_reproduction["Notebook_07"]
    - metric_reproduction["Notebook_04"]
).abs()

display(metric_reproduction)

In [ ]:
prediction_results = test_metadata.copy()

prediction_results["Actual_CR_FIQA"] = y_test.to_numpy()
prediction_results["Predicted_CR_FIQA"] = test_predictions
prediction_results["Residual"] = (
    prediction_results["Actual_CR_FIQA"]
    - prediction_results["Predicted_CR_FIQA"]
)
prediction_results["Absolute_Error"] = (
    prediction_results["Residual"].abs()
)

display(
    prediction_results
    .sort_values("Absolute_Error", ascending=False)
    .head(10)
)

## 6. Repeated permutation importance

Permutation importance measures how much held-out RMSE increases when one
original input feature is shuffled while all other inputs remain unchanged.

- larger positive values indicate greater predictive contribution;
- values near zero indicate little measurable contribution;
- negative values can arise from noise, correlated predictors, or sampling
  variation.

Because permutation is applied to the full pipeline, the results remain on the
original project-feature level.

In [ ]:
def negative_rmse_scorer(model, X, y):
    predictions = model.predict(X)

    return -(
        mean_squared_error(
            y,
            predictions,
        ) ** 0.5
    )


permutation_result = permutation_importance(
    final_model,
    X_test,
    y_test,
    scoring=negative_rmse_scorer,
    n_repeats=PERMUTATION_REPEATS,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

permutation_table = pd.DataFrame(
    {
        "Feature": feature_columns,
        "Permutation_Importance_Mean": (
            permutation_result.importances_mean
        ),
        "Permutation_Importance_SD": (
            permutation_result.importances_std
        ),
    }
)

permutation_table["Permutation_Rank"] = (
    permutation_table["Permutation_Importance_Mean"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

permutation_table = (
    permutation_table
    .sort_values(
        "Permutation_Importance_Mean",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(permutation_table)

In [ ]:
permutation_plot_data = (
    permutation_table
    .head(TOP_N_FEATURES)
    .sort_values("Permutation_Importance_Mean")
)

plt.figure(figsize=(9, 7))
plt.barh(
    permutation_plot_data["Feature"],
    permutation_plot_data["Permutation_Importance_Mean"],
    xerr=permutation_plot_data["Permutation_Importance_SD"],
    capsize=3,
)
plt.axvline(0, linewidth=1)
plt.xlabel("Increase in test RMSE after permutation")
plt.ylabel("Feature")
plt.title("Repeated Permutation Importance")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "permutation_importance.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 7. Separate preprocessing and estimator for SHAP

The saved object is a complete scikit-learn pipeline. SHAP explains the final
estimator on the exact transformed representation created by the fitted
preprocessor.

In [ ]:
def extract_estimator_and_preprocessor(model_object):
    if not isinstance(model_object, Pipeline):
        return model_object, None

    estimator = model_object.steps[-1][1]

    if len(model_object.steps) == 1:
        return estimator, None

    preprocessor = Pipeline(model_object.steps[:-1])

    return estimator, preprocessor


estimator, preprocessor = extract_estimator_and_preprocessor(
    final_model
)

print("Estimator:", type(estimator))
print(
    "Preprocessor:",
    type(preprocessor) if preprocessor is not None else None,
)

In [ ]:
def transform_features(
    preprocessor_object,
    X,
    original_feature_names,
):
    if preprocessor_object is None:
        return X.copy()

    transformed = preprocessor_object.transform(X)

    if hasattr(transformed, "toarray"):
        transformed = transformed.toarray()

    try:
        transformed_names = (
            preprocessor_object.get_feature_names_out(
                original_feature_names
            )
        )
    except Exception:
        transformed_names = [
            f"feature_{index}"
            for index in range(transformed.shape[1])
        ]

    return pd.DataFrame(
        transformed,
        index=X.index,
        columns=transformed_names,
    )


X_test_transformed = transform_features(
    preprocessor,
    X_test,
    feature_columns,
)

print(
    "Transformed test shape:",
    X_test_transformed.shape,
)
print(
    "Transformed features:",
    X_test_transformed.columns.tolist(),
)

## 8. Select a reproducible SHAP sample

The fitted model is not changed. Only the number of test observations explained
by SHAP is limited when necessary to keep runtime and memory use manageable.

In [ ]:
if len(X_test_transformed) > MAX_SHAP_ROWS:
    shap_X = X_test_transformed.sample(
        n=MAX_SHAP_ROWS,
        random_state=RANDOM_STATE,
    )
else:
    shap_X = X_test_transformed.copy()

print(f"Rows used for SHAP: {len(shap_X):,}")

## 9. Calculate SHAP values

In [ ]:
explainer = shap.Explainer(
    estimator,
    shap_X,
)

try:
    shap_values = explainer(
        shap_X,
        check_additivity=False,
    )
except TypeError:
    shap_values = explainer(shap_X)

if shap_values.values.ndim != 2:
    raise ValueError(
        "Expected a two-dimensional SHAP value matrix for regression."
    )

print("SHAP value shape:", shap_values.values.shape)

## 10. Global SHAP importance on transformed features

Global SHAP importance is the mean absolute SHAP value. It measures how strongly
a transformed feature changes predictions on average. It does not by itself
show a causal effect or a universally positive or negative direction.

In [ ]:
transformed_shap_importance = pd.DataFrame(
    {
        "Transformed_Feature": shap_X.columns,
        "Mean_Absolute_SHAP": np.abs(
            shap_values.values
        ).mean(axis=0),
        "Mean_Signed_SHAP": (
            shap_values.values.mean(axis=0)
        ),
    }
)

transformed_shap_importance["Transformed_SHAP_Rank"] = (
    transformed_shap_importance["Mean_Absolute_SHAP"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

transformed_shap_importance = (
    transformed_shap_importance
    .sort_values(
        "Mean_Absolute_SHAP",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(transformed_shap_importance.head(20))

## 11. Map transformed SHAP values back to original project features

Categorical preprocessing can expand one original feature into several encoded
columns. To compare SHAP with RQ2, RQ3, and permutation importance, transformed
SHAP magnitudes are aggregated back to the original feature level.

In [ ]:
def map_transformed_to_original(
    transformed_feature_name,
    original_features,
):
    if transformed_feature_name in original_features:
        return transformed_feature_name

    cleaned_name = transformed_feature_name

    if "__" in cleaned_name:
        cleaned_name = cleaned_name.split("__", 1)[1]

    exact_matches = [
        feature
        for feature in original_features
        if cleaned_name == feature
    ]

    if exact_matches:
        return exact_matches[0]

    prefix_matches = [
        feature
        for feature in original_features
        if cleaned_name.startswith(f"{feature}_")
    ]

    if prefix_matches:
        return max(prefix_matches, key=len)

    return cleaned_name


transformed_to_original = {
    column: map_transformed_to_original(
        column,
        feature_columns,
    )
    for column in shap_X.columns
}

feature_mapping_table = pd.DataFrame(
    {
        "Transformed_Feature": list(
            transformed_to_original.keys()
        ),
        "Original_Feature": list(
            transformed_to_original.values()
        ),
    }
)

display(feature_mapping_table)

In [ ]:
absolute_shap = pd.DataFrame(
    np.abs(shap_values.values),
    index=shap_X.index,
    columns=shap_X.columns,
)

signed_shap = pd.DataFrame(
    shap_values.values,
    index=shap_X.index,
    columns=shap_X.columns,
)

original_feature_shap_rows = []

for original_feature in feature_columns:
    transformed_columns = [
        transformed_feature
        for transformed_feature, mapped_feature
        in transformed_to_original.items()
        if mapped_feature == original_feature
    ]

    if not transformed_columns:
        continue

    rowwise_absolute_sum = (
        absolute_shap[transformed_columns]
        .sum(axis=1)
    )

    rowwise_signed_sum = (
        signed_shap[transformed_columns]
        .sum(axis=1)
    )

    original_feature_shap_rows.append(
        {
            "Feature": original_feature,
            "Transformed_Columns": len(
                transformed_columns
            ),
            "Mean_Absolute_SHAP": (
                rowwise_absolute_sum.mean()
            ),
            "Mean_Signed_SHAP": (
                rowwise_signed_sum.mean()
            ),
        }
    )

shap_importance_table = pd.DataFrame(
    original_feature_shap_rows
)

shap_importance_table["SHAP_Rank"] = (
    shap_importance_table["Mean_Absolute_SHAP"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

shap_importance_table = (
    shap_importance_table
    .sort_values(
        "Mean_Absolute_SHAP",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(shap_importance_table)

## 12. Global SHAP plots

### 12.1 SHAP bar plot

The standard SHAP bar plot shows importance on the transformed feature level.
The aggregated original-feature table above should be used for cross-notebook
comparisons.

In [ ]:
shap.plots.bar(
    shap_values,
    max_display=TOP_N_FEATURES,
    show=False,
)

plt.title("Global SHAP Feature Importance")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "shap_global_bar.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

### 12.2 SHAP beeswarm plot

- positive SHAP values increase the predicted CR-FIQA score;
- negative SHAP values decrease it;
- color represents the transformed feature value;
- each point represents one explained test observation.

In [ ]:
shap.plots.beeswarm(
    shap_values,
    max_display=TOP_N_FEATURES,
    show=False,
)

plt.title("SHAP Beeswarm Plot")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "shap_beeswarm.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 13. SHAP dependence plots

Dependence plots are created for the strongest transformed SHAP features. They
can reveal nonlinearity, thresholds, and interaction-like patterns that are not
visible in a single global ranking.

In [ ]:
top_dependence_features = (
    transformed_shap_importance[
        "Transformed_Feature"
    ]
    .head(TOP_N_DEPENDENCE_PLOTS)
    .tolist()
)

print("Dependence plots for:")
print(top_dependence_features)

In [ ]:
for feature in top_dependence_features:
    shap.plots.scatter(
        shap_values[:, feature],
        color=shap_values,
        show=False,
    )

    plt.title(f"SHAP Dependence: {feature}")
    plt.tight_layout()

    safe_name = (
        feature
        .replace("/", "_")
        .replace(" ", "_")
        .replace(":", "_")
    )

    plt.savefig(
        FIGURES_PATH
        / f"shap_dependence_{safe_name}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

## 14. Local explanations

Two observations from the SHAP sample are explained:

1. the observation with the smallest absolute prediction error;
2. the observation with the largest absolute prediction error.

This demonstrates both a successful and a difficult prediction without
generating repetitive plots for many individual images.

In [ ]:
prediction_lookup = prediction_results.copy()
prediction_lookup.index = X_test.index

available_predictions = prediction_lookup.loc[
    prediction_lookup.index.intersection(
        shap_X.index
    )
].copy()

if available_predictions.empty:
    raise RuntimeError(
        "No SHAP rows could be matched to prediction metadata."
    )

best_prediction_index = (
    available_predictions["Absolute_Error"].idxmin()
)

worst_prediction_index = (
    available_predictions["Absolute_Error"].idxmax()
)

local_cases = {
    "best_prediction": best_prediction_index,
    "worst_prediction": worst_prediction_index,
}

shap_index_lookup = pd.Series(
    np.arange(len(shap_X)),
    index=shap_X.index,
)

local_summary_rows = []

In [ ]:
for case_name, original_index in local_cases.items():
    shap_position = int(
        shap_index_lookup.loc[original_index]
    )

    row_information = prediction_lookup.loc[
        original_index
    ]

    local_summary_rows.append(
        {
            "Case": case_name,
            IMAGE_COLUMN: row_information[IMAGE_COLUMN],
            IDENTITY_COLUMN: row_information[IDENTITY_COLUMN],
            GROUP_COLUMN: row_information[GROUP_COLUMN],
            "Actual_CR_FIQA": row_information[
                "Actual_CR_FIQA"
            ],
            "Predicted_CR_FIQA": row_information[
                "Predicted_CR_FIQA"
            ],
            "Residual": row_information["Residual"],
            "Absolute_Error": row_information[
                "Absolute_Error"
            ],
        }
    )

    shap.plots.waterfall(
        shap_values[shap_position],
        max_display=TOP_N_FEATURES,
        show=False,
    )

    readable_case_name = case_name.replace("_", " ").title()

    plt.title(f"Local SHAP Explanation — {readable_case_name}")
    plt.tight_layout()
    plt.savefig(
        FIGURES_PATH
        / f"shap_waterfall_{case_name}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

local_explanations_summary = pd.DataFrame(
    local_summary_rows
)

display(local_explanations_summary)

## 15. Compare RQ2, RQ3, permutation importance, and SHAP

The four analyses answer different questions:

- **RQ2:** Is a feature associated with CR-FIQA on its own?
- **RQ3:** Does the association remain after linear adjustment?
- **Permutation importance:** Does shuffling the original feature worsen
  held-out prediction?
- **SHAP:** How strongly does the final nonlinear model use the feature across
  explained observations?

Agreement across methods strengthens interpretation, but disagreement is also
informative and may reflect nonlinearity, interactions, collinearity, or weak
univariate effects.

In [ ]:
comparison_table = (
    shap_importance_table
    .merge(
        permutation_table,
        on="Feature",
        how="outer",
        validate="one_to_one",
    )
)

if RQ2_FILE.exists():
    rq2_results = pd.read_csv(RQ2_FILE)

    rq2_columns = [
        column
        for column in [
            "Feature",
            "Pearson_R",
            "Pearson_FDR_P",
            "Pearson_Significant_FDR",
            "Spearman_Rho",
            "Spearman_FDR_P",
            "Spearman_Significant_FDR",
            "Both_Significant_FDR",
        ]
        if column in rq2_results.columns
    ]

    comparison_table = comparison_table.merge(
        rq2_results[rq2_columns],
        on="Feature",
        how="left",
        validate="one_to_one",
    )
else:
    print(
        "RQ2 table not found; comparison will omit correlation results."
    )

if RQ3_FILE.exists():
    rq3_results = pd.read_csv(RQ3_FILE)

    rq3_results = rq3_results.loc[
        rq3_results["Feature"] != "const"
    ].copy()

    rq3_results = rq3_results.rename(
        columns={
            "Coefficient": "RQ3_HC3_Coefficient",
            "P_Value": "RQ3_P_Value",
            "Adjusted_P_Value_FDR": "RQ3_FDR_P",
            "Significant_FDR": "RQ3_Significant_FDR",
        }
    )

    rq3_columns = [
        column
        for column in [
            "Feature",
            "RQ3_HC3_Coefficient",
            "RQ3_P_Value",
            "RQ3_FDR_P",
            "RQ3_Significant_FDR",
        ]
        if column in rq3_results.columns
    ]

    comparison_table = comparison_table.merge(
        rq3_results[rq3_columns],
        on="Feature",
        how="left",
        validate="one_to_one",
    )
else:
    print(
        "RQ3 table not found; comparison will omit regression results."
    )

comparison_table = (
    comparison_table
    .sort_values(
        "SHAP_Rank",
        na_position="last",
    )
    .reset_index(drop=True)
)

display(comparison_table)

## 16. Rank agreement

In [ ]:
rank_agreement_table = comparison_table[
    [
        column
        for column in [
            "Feature",
            "SHAP_Rank",
            "Permutation_Rank",
        ]
        if column in comparison_table.columns
    ]
].copy()

if {
    "SHAP_Rank",
    "Permutation_Rank",
}.issubset(rank_agreement_table.columns):
    complete_rank_rows = rank_agreement_table.dropna(
        subset=["SHAP_Rank", "Permutation_Rank"]
    )

    if len(complete_rank_rows) >= 3:
        rank_correlation = complete_rank_rows[
            ["SHAP_Rank", "Permutation_Rank"]
        ].corr(method="spearman").iloc[0, 1]
    else:
        rank_correlation = np.nan

    rank_agreement_table["Absolute_Rank_Difference"] = (
        rank_agreement_table["SHAP_Rank"]
        - rank_agreement_table["Permutation_Rank"]
    ).abs()
else:
    rank_correlation = np.nan

print(
    "Spearman agreement between SHAP and permutation ranks:",
    rank_correlation,
)

display(
    rank_agreement_table.sort_values(
        "Absolute_Rank_Difference"
        if "Absolute_Rank_Difference"
        in rank_agreement_table.columns
        else "Feature"
    )
)

## 17. Focused explainability summary

In [ ]:
focused_columns = [
    column
    for column in [
        "Feature",
        "SHAP_Rank",
        "Mean_Absolute_SHAP",
        "Mean_Signed_SHAP",
        "Permutation_Rank",
        "Permutation_Importance_Mean",
        "Permutation_Importance_SD",
        "Pearson_R",
        "Pearson_FDR_P",
        "Pearson_Significant_FDR",
        "Spearman_Rho",
        "Spearman_FDR_P",
        "Spearman_Significant_FDR",
        "RQ3_HC3_Coefficient",
        "RQ3_FDR_P",
        "RQ3_Significant_FDR",
    ]
    if column in comparison_table.columns
]

focused_explainability_summary = (
    comparison_table[focused_columns]
    .sort_values(
        "SHAP_Rank",
        na_position="last",
    )
    .reset_index(drop=True)
)

display(focused_explainability_summary)

## 18. Automated interpretation flags

In [ ]:
interpretation_table = comparison_table.copy()

if "Pearson_Significant_FDR" in interpretation_table.columns:
    interpretation_table["Supported_by_RQ2"] = (
        interpretation_table[
            "Pearson_Significant_FDR"
        ].fillna(False)
    )
else:
    interpretation_table["Supported_by_RQ2"] = False

if "RQ3_Significant_FDR" in interpretation_table.columns:
    interpretation_table["Supported_by_RQ3"] = (
        interpretation_table[
            "RQ3_Significant_FDR"
        ].fillna(False)
    )
else:
    interpretation_table["Supported_by_RQ3"] = False

interpretation_table["Top_SHAP_Feature"] = (
    interpretation_table["SHAP_Rank"]
    <= TOP_N_FEATURES
)

interpretation_table["Top_Permutation_Feature"] = (
    interpretation_table["Permutation_Rank"]
    <= TOP_N_FEATURES
)

interpretation_table["Cross_Method_Support_Count"] = (
    interpretation_table[
        [
            "Supported_by_RQ2",
            "Supported_by_RQ3",
            "Top_SHAP_Feature",
            "Top_Permutation_Feature",
        ]
    ]
    .astype(int)
    .sum(axis=1)
)

interpretation_table = (
    interpretation_table
    .sort_values(
        [
            "Cross_Method_Support_Count",
            "SHAP_Rank",
        ],
        ascending=[False, True],
        na_position="last",
    )
    .reset_index(drop=True)
)

display(
    interpretation_table[
        [
            "Feature",
            "Cross_Method_Support_Count",
            "Supported_by_RQ2",
            "Supported_by_RQ3",
            "Top_SHAP_Feature",
            "Top_Permutation_Feature",
        ]
    ]
)

## 19. Reporting guidance

For the written results section, report:

1. whether Notebook 07 reproduced the Notebook 04 metrics;
2. the highest-ranked original features by SHAP;
3. the highest-ranked original features by permutation importance;
4. the degree of agreement between both predictive rankings;
5. nonlinear or threshold patterns visible in dependence plots;
6. whether important predictive features were also supported by RQ2 or RQ3;
7. notable differences between the best- and worst-predicted local examples.

Use cautious wording such as:

> Higher values of the feature were associated with negative SHAP contributions
> and lower model predictions.

Do not write:

> The feature caused lower CR-FIQA scores.

SHAP values explain the fitted model, not the data-generating process. Correlated
features can share or redistribute importance, and global importance does not
guarantee the same effect for every observation.

## 20. Save tables and metadata

In [ ]:
performance_summary.to_csv(
    TABLES_PATH / "model_performance.csv",
    index=False,
)

metric_reproduction.to_csv(
    TABLES_PATH / "metric_reproduction_check.csv",
    index=False,
)

prediction_results.to_csv(
    TABLES_PATH / "test_predictions.csv",
    index=False,
)

permutation_table.to_csv(
    TABLES_PATH / "permutation_importance.csv",
    index=False,
)

transformed_shap_importance.to_csv(
    TABLES_PATH / "transformed_shap_importance.csv",
    index=False,
)

feature_mapping_table.to_csv(
    TABLES_PATH / "transformed_to_original_feature_mapping.csv",
    index=False,
)

shap_importance_table.to_csv(
    TABLES_PATH / "shap_importance_original_features.csv",
    index=False,
)

local_explanations_summary.to_csv(
    TABLES_PATH / "local_explanations_summary.csv",
    index=False,
)

comparison_table.to_csv(
    TABLES_PATH / "rq2_rq3_explainability_comparison.csv",
    index=False,
)

rank_agreement_table.to_csv(
    TABLES_PATH / "rank_agreement.csv",
    index=False,
)

focused_explainability_summary.to_csv(
    TABLES_PATH / "focused_explainability_summary.csv",
    index=False,
)

interpretation_table.to_csv(
    TABLES_PATH / "cross_method_interpretation.csv",
    index=False,
)

print("All Notebook 07 tables were saved.")

In [ ]:
run_metadata = {
    "random_state": RANDOM_STATE,
    "permutation_repeats": PERMUTATION_REPEATS,
    "maximum_shap_rows": MAX_SHAP_ROWS,
    "shap_rows_used": int(len(shap_X)),
    "final_model_name": optimization_summary.get(
        "final_model"
    ),
    "test_rows": int(len(X_test)),
    "feature_columns": feature_columns,
    "test_metrics": {
        "rmse": float(test_rmse),
        "mae": float(test_mae),
        "r2": float(test_r2),
    },
    "shap_permutation_rank_spearman": (
        None
        if pd.isna(rank_correlation)
        else float(rank_correlation)
    ),
}

with (
    RESULTS_PATH / "run_metadata.json"
).open("w", encoding="utf-8") as file:
    json.dump(run_metadata, file, indent=2)

print("Saved Notebook 07 metadata.")

## 21. Saved-file summary

In [ ]:
print("Notebook 07 result files:")

for file_path in sorted(RESULTS_PATH.rglob("*")):
    if file_path.is_file():
        print("-", file_path.relative_to(RESULTS_PATH))

## 22. Final result and transition

Notebook 07 provides the model-explanation layer of the project:

- Notebook 05: univariate statistical association;
- Notebook 06: adjusted linear association;
- Notebook 07: predictive reliance of the final nonlinear model.

The central output for the paper is:

```text
results/07_model_explainability/tables/
focused_explainability_summary.csv
```

It combines SHAP, permutation importance, RQ2, and RQ3 in one feature-level
table. Demographic consistency and fairness analyses should remain separate so
that model explainability is not confused with group fairness.